## Step5-Sense-Checking

Makes a few key plots of the final subglacial discharge files to check for possible bugs.

Creator: Donald Slater, donald.slater@ed.ac.uk

Cleaned up by Donald Slater, 13 Mar 2026.

#### Required files/inputs
- CMIP monthly per-basin bias-corrected subglacial discharge files, split into annual files e.g. *Q_CESM2-WACCM_historical_SDBN1_1850.nc* (produced in step 4)
- RACMO monthly 1958-2024 per-basin subglacial discharge Q_RACMO_1958_2024.nc (produced in step 1)

#### Outputs
- Maps of subglacial discharge and subglacial discharge change over 20-yr time periods
- Annual time series of subglacial discharge from 4 large glaciers
- Seasonality of subglacial discharge for the same glaciers
- Plot to check the bias correction

In [1]:
# imports
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
# inputs
cmip_model = 'CESM2-WACCM' # cmip model
scenario = 'ssp585' # scenario

# directories
hist_dir = '/Users/dslater2/Documents/' + cmip_model + '/historical/runoff/' # historical files
proj_dir = '/Users/dslater2/Documents/' + cmip_model + '/' + scenario + '/runoff/projection/' # ssp files to 2100
ext_dir = '/Users/dslater2/Documents/' + cmip_model + '/' + scenario + '/runoff/extension/' # ssp files to 2300
plot_dir = '/Users/dslater2/Documents/' + cmip_model + '/plots/' # directory to save plots

# racmo file for checking bias correction
racmo_runoff_file = '/Users/dslater2/Documents/RACMO/Q_RACMO_1958_2024.nc'

# major glacier positions to extract time series (meters)
xpos = [-155675,  288025,   -236675,  426175]
ypos = [-2274280, -2573080, -1035880, -1083580]
namepos = ['Jakobshavn', 'Helheim', 'Petermann', '79N']
colors = ["tab:blue", "tab:orange", "tab:green", "tab:red"]

### Make some map plots to check spatial patterns, general magnitude and differences.

Here's an example of what we'd expect to see. The figure contains maps of 20-yr mean subglacial discharge for three time periods (top row) and their differences relative to 1985–2014 (bottom row) for CESM2-WACCM under ssp585. Panel d shows the locations of the 4 time series in the next two plots.

<img src="./plots_CESM2-WACCM_ssp585_sgd_map.png" width="800">

Looking for
- Correct x and y axes (units are km).
- Sensible magnitudes for sgd. That is, probably 0-800 m3/s in 1985-2014 (in fact all climate models should look very similar to panel a of this plot if the bias correction has worked well), larger (roughly a factor 4 in CESM2-WACCM ssp585) by the end of the century and larger again by 2300. Obviously would expect different change for different scenarios.
- Spatial patterns make sense: there should be no NaN values, the region beyond the continental shelf should be uniform, the basins should look correct and stay consistent through the time series.

In [ ]:
# map plots of discharge and discharge change

# periods
start_yrs = [1985, 2081, 2281]
end_yrs   = [2014, 2100, 2300]
means = [] 

for i, (start, end) in enumerate(zip(start_yrs, end_yrs)):

    Qs = []

    for year in range(start, end + 1):
        if year <=2014:
            file = hist_dir + 'Q_' + cmip_model + '_historical_SDBN1_' + str(year) + '.nc'
        elif year<=2100:
            file = proj_dir + 'Q_' + cmip_model + '_ssp585_SDBN1_' + str(year) + '.nc'
        else:
            file = ext_dir + 'Q_' + cmip_model + '_ssp585_SDBN1_' + str(year) + '.nc'
        ds = xr.open_dataset(file)

        if year == start:
            x = ds["x"]/1e3 # in km
            y = ds["y"]/1e3 # in km

        Qs.append(ds["Q"].mean(axis=0))  # annual mean

    # mean over all years in period
    Q_mean = xr.concat(Qs, dim="year").mean("year")
    means.append(Q_mean)

# --- plotting ---
fig, axes = plt.subplots(2, 3, figsize=(12, 9))

# row 1: means
for i in range(3):
    pcm = axes[0, i].pcolormesh(x, y, means[i], shading="auto", cmap="viridis")
    axes[0, i].set_title(f"{start_yrs[i]}-{end_yrs[i]} mean sgd (m3/s)")
    axes[0, i].set_aspect("equal")
    fig.colorbar(pcm, ax=axes[0, i])

# row 2: differences relative to first period
for i in range(3):
    diff = means[i] - means[0]
    colmax = float(np.max(np.abs(diff)))
    if colmax==0:
        colmax=1
    pcm = axes[1, i].pcolormesh(x, y, diff, vmin=-colmax, vmax=colmax, shading="auto", cmap="RdBu_r")
    if i==0:
        for xnow, ynow, c, name in zip(np.array(xpos), np.array(ypos), colors, namepos):
            axes[1, i].plot(xnow/1e3, ynow/1e3, 'o', color=c, markersize=10, label=name)
        axes[1, i].legend()
    axes[1, i].set_title(f"{start_yrs[i]}-{end_yrs[i]} vs {start_yrs[0]}-{end_yrs[0]} sgd (m3/s)")
    axes[1, i].set_aspect("equal")
    fig.colorbar(pcm, ax=axes[1, i])

fig.suptitle(cmip_model+'_'+scenario, fontsize=16)
plt.tight_layout()
plt.savefig(plot_dir+'plots_'+cmip_model+'_'+scenario+'_sgd_map.png', dpi=300, bbox_inches="tight")
plt.show()

### Time series plots to check general magnitude and trends

Here's an example of what we'd expect to see. The figure contains time series of annual mean sgd, extracted from points in the drainage basin of 4 major glaciers, for CESM2-WACCM in a ssp585 scenario. We separately plot 1850-2014 and 2015-2300 since the scales can be very different.

<img src="./plots_CESM2-WACCM_ssp585_sgd_annualtimeseries.png" width="800">

Looking for
- Sensible magnitudes for sgd. That is, probably 0-400 m3/s in 1985-2014 for the 4 glaciers (in fact all climate models should look similar to panel a of this plot if the bias correction has worked well) then increasing significantly in warm climate scenarios. Obviously would expect different change for different scenarios.
- Temporal patterns broadly consistent with climate scenario, e.g., increasing, increasing by different amounts depending on scenario, stabilisation or later decrease in overshoot scenarios.

In [ ]:
# time series plots - annual

# Period
start_yr = 1850
end_yr   = 2300

# Initialize storage
Q_monthly = {f"point_{i}": [] for i in range(len(xpos))}
Q_annual  = {f"point_{i}": [] for i in range(len(xpos))}
time_all  = []

# Loop over years
for year in range(start_yr, end_yr + 1):

    if year <= 2014:
        file = f"{hist_dir}/Q_{cmip_model}_historical_SDBN1_{year}.nc"
    elif year <= 2100:
        file = f"{proj_dir}/Q_{cmip_model}_ssp585_SDBN1_{year}.nc"
    else:
        file = f"{ext_dir}/Q_{cmip_model}_ssp585_SDBN1_{year}.nc"

    with xr.open_dataset(file) as ds:

        # Extract x/y coordinates once
        if year == start_yr:
            x = ds["x"].values
            y = ds["y"].values

            # Find nearest indices for all points
            indices = [(np.abs(x - xp).argmin(), np.abs(y - yp).argmin()) for xp, yp in zip(xpos, ypos)]

        # Store time once per file
        time_all.append(ds["time"].values)

        # Extract Q for each point
        for i, (ix, iy) in enumerate(indices):
            Q_month = ds["Q"][:, iy, ix].values  # monthly values
            Q_monthly[f"point_{i}"].append(Q_month)
            Q_annual[f"point_{i}"].append(Q_month.mean())  # annual mean

# Convert lists to arrays
for i in range(len(xpos)):
    Q_monthly[f"point_{i}"] = np.concatenate(Q_monthly[f"point_{i}"])  # shape (n_months_total,)
    Q_annual[f"point_{i}"] = np.array(Q_annual[f"point_{i}"])          # shape (n_years_total,)

# Flatten time array
time_index = np.concatenate(time_all)

# annual mean plots
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Convert cftime to decimal year
def cftime_to_year(t):
    return t.year + (t.month - 1)/12 + (t.day - 1)/365
years_decimal = np.array([cftime_to_year(t) for t in time_index])
years_annual = np.floor(years_decimal)[::12]

# historical
mask_hist = (years_annual >= 1850) & (years_annual <= 2014)
for i in range(len(xpos)):
    axes[0].plot(years_annual[mask_hist], Q_annual[f"point_{i}"][mask_hist],label=namepos[i], linewidth=1.5)
axes[0].set_title('Annual mean sgd, historical')
axes[0].set_xlabel("Year")
axes[0].set_ylabel("m3/s")
axes[0].grid(True)
axes[0].set_xlim(1850,2014)

# projection
mask_proj = (years_annual >= 2015) & (years_annual <= 2300)
for i in range(len(xpos)):
    axes[1].plot(years_annual[mask_proj], Q_annual[f"point_{i}"][mask_proj],label=namepos[i], linewidth=1.5)
axes[1].set_title('Annual mean sgd, projection')
axes[1].set_xlabel("Year")
axes[1].set_ylabel("m3/s")
axes[1].grid(True)
axes[1].legend(loc='upper left')
axes[1].set_xlim(2015,2300)

fig.suptitle(cmip_model+'_'+scenario, fontsize=16)
plt.tight_layout()
plt.savefig(plot_dir+'plots_'+cmip_model+'_'+scenario+'_sgd_annualtimeseries.png', dpi=300, bbox_inches="tight")
plt.show()

### Seasonal plots

Here's an example of roughly what we'd expect to see. The figure shows the seasonality in sgd at the same 4 major glaciers. Each panel shows a different 20-yr time period. The solid line shows the mean sgd during that time period in a given month, the shading shows the min-max range of sgd during that time period in a given month. This is for CESM2-WACCM in a ssp585 scenario.

<img src="./plots_CESM2-WACCM_ssp585_sgd_seasonal.png" width="800">

If the bias-correction has worked well, then panel a should look similar for all climate models. We should see that the vast majority of sgd is in June, July and August with only a little outside of that. In warm climate scenarios, both the overall magnitude and the length of the melt season will increase.

In [ ]:
# check seasonality

# Periods
start_yrs = [1985, 2081, 2281]
end_yrs   = [2014, 2100, 2300]

# Create figure with 3 subplots in a row
fig, axes = plt.subplots(1, 3, figsize=(12, 5))

months = np.arange(1, 13)

for ax, (start, end) in zip(axes, zip(start_yrs, end_yrs)):
    for j, point in enumerate(namepos):
        # Mask for current period
        mask = (years_decimal >= start) & (years_decimal <= end)
        Q_period = Q_monthly[f"point_{j}"][mask]

        # Compute monthly min, max, mean
        Qmin = np.zeros(12)
        Qmax = np.zeros(12)
        Qmean = np.zeros(12)
        for i in range(12):
            month_vals = Q_period[i::12]
            Qmin[i] = month_vals.min()
            Qmax[i] = month_vals.max()
            Qmean[i] = month_vals.mean()

        # Shaded min-max range
        ax.fill_between(months, Qmin, Qmax, color=colors[j], alpha=0.2)

        # Solid mean line
        ax.plot(months, Qmean, color=colors[j], linewidth=2, label=point)

    # Customize subplot
    ax.set_xticks(months)
    ax.set_xlabel("Month")
    ax.set_title(f"{start}-{end}")
    ax.grid(True)

# Add y-label to leftmost subplot
axes[0].set_ylabel("Monthly subglacial discharge (m3/s)")

# Legend (only once, using the first subplot)
axes[0].legend(loc='upper left')

fig.suptitle(cmip_model+'_'+scenario, fontsize=16)
plt.tight_layout()  # leave space for suptitle
plt.savefig(plot_dir+'plots_'+cmip_model+'_'+scenario+'_sgd_seasonal.png', dpi=300, bbox_inches="tight")
plt.show()

### Check bias correction

Here's an example of what this should look like, in this case for CESM2-WACCM ssp585 (in fact given that this only uses 1985-2014, which is in the historical period, it should be the same for all projection scenarios). The lines show the mean seasonality (i.e., the mean 1985-2014 sgd in a given month) from the bias-corrected CMIP model. The dots show the same quantity but from RACMO.

<img src="./plots_CESM2-WACCM_ssp585_sgd_biascorrection.png" width="800">

The dots and the lines should be very close if the bias-correction has worked. Some small deviations (a few percent) can be expected because sometimes the bias-correction could take a given month below 0. Since we don't allow sgd values below 0, these are reset to 0 and when the mean is recalculated, there can therefore be small differences.

In [ ]:
# check bias correction

# read data for same positions from racmo
ds = xr.open_dataset(racmo_runoff_file,decode_times=False)

# Assuming the dataset has variables: 'Q', 'x', 'y', 'time'
x = ds["x"].values  # in m or km
y = ds["y"].values
time = ds["time"].values  # months since jan 1850
years_decimal_racmo = 1850+time/12

# Create dictionary to store Q at each point
Q_racmo_monthly = {}

for j, (xi, yi) in enumerate(zip(xpos, ypos)):
    # Find the nearest grid point indices
    ix = np.abs(x - xi).argmin()
    iy = np.abs(y - yi).argmin()
    
    # Extract the time series for this point (all months)
    Q_racmo_monthly[f"point_{j}"] = ds["Q"][:, iy, ix].values  # shape: time

# Period
start_yr = 1985
end_yr = 2014

# Colors for each glacier
colors = ["tab:blue", "tab:orange", "tab:green", "tab:red"]

# Mask for period
mask = (years_decimal >= start_yr) & (years_decimal <= end_yr)
mask_racmo = (years_decimal_racmo >= start_yr) & (years_decimal_racmo <= end_yr)

months = np.arange(1, 13)
plt.figure(figsize=(12,5))

# Loop over glaciers
for j, point in enumerate(namepos):
    Q_period = Q_monthly[f"point_{j}"][mask]
    Q_racmo_period = Q_racmo_monthly[f"point_{j}"][mask_racmo]

    # Compute mean for each month
    Qmean = np.zeros(12)
    Qmean_racmo = np.zeros(12)
    for i in range(12):
        month_vals = Q_period[i::12]
        Qmean[i] = month_vals.mean()
        month_vals_racmo = Q_racmo_period[i::12]
        Qmean_racmo[i] = month_vals_racmo.mean()

    # Plot solid mean line
    plt.plot(months, Qmean, color=colors[j], linewidth=2, label=point+' (bias-corrected CMIP)', alpha=0.5)
    plt.plot(months, Qmean_racmo, color=colors[j], linewidth=0, marker='o', label=point+' (RACMO)')

# Final touches
plt.xticks(months)
plt.xlabel("Month")
plt.ylabel("Monthly subglacial discharge (m3/s)")
plt.title(f"Over bias correction period ({start_yr}-{end_yr})")
plt.grid(True)
plt.suptitle(cmip_model+'_'+scenario, fontsize=16)
plt.legend(loc='upper left')
plt.savefig(plot_dir+'plots_'+cmip_model+'_'+scenario+'_sgd_biascorrection.png', dpi=300, bbox_inches="tight")
plt.show()